In [ ]:
# ========================================
# Cell 1: PixelRNN Family Models (Paper-Accurate Implementation - FAIR COMPARISON)
# ========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# -----------------------------
# Masked Convolution with Proper RGB Channel Dependencies
# -----------------------------
class MaskedConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, mask_type, **kwargs):
        super().__init__(in_channels, out_channels, kernel_size, **kwargs)
        assert mask_type in ["A", "B"]
        self.mask_type = mask_type
        self.register_buffer("mask", torch.ones_like(self.weight))
        
        _, _, h, w = self.weight.size()
        yc, xc = h // 2, w // 2
        
        # Zero out future pixels (below and to the right)
        self.mask[:, :, yc+1:, :] = 0
        self.mask[:, :, yc, xc+1:] = 0
        
        # For mask type A, also zero out the center pixel
        if mask_type == "A":
            self.mask[:, :, yc, xc] = 0
            
        # Handle RGB channel dependencies properly
        if in_channels >= 3 and out_channels >= 3:
            for out_c in range(3):
                for in_c in range(out_c + (1 if mask_type == "B" else 0), 3):
                    out_start = (out_channels // 3) * out_c
                    out_end = (out_channels // 3) * (out_c + 1)
                    in_start = (in_channels // 3) * in_c  
                    in_end = (in_channels // 3) * (in_c + 1)
                    self.mask[out_start:out_end, in_start:in_end, yc, xc] = 0

    def forward(self, x):
        masked_weight = self.weight * self.mask
        return F.conv2d(x, masked_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

# -----------------------------
# Residual Block for PixelRNN
# -----------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels // 2, 1)
        self.conv2 = MaskedConv2d(channels // 2, channels // 2, 3, 'B', padding=1)
        self.conv3 = nn.Conv2d(channels // 2, channels, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.relu(self.conv2(out))
        out = self.conv3(out)
        return residual + out

# -----------------------------
# PixelCNN (Paper Specification - Table 1)
# -----------------------------
class PixelCNN(nn.Module):
    def __init__(self, in_channels=3, hidden=128, n_layers=15):
        super().__init__()
        self.in_channels = in_channels
        
        # First layer: 7x7 masked conv with type A
        self.first_conv = MaskedConv2d(in_channels, hidden, 7, 'A', padding=3)
        
        # Residual blocks
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden) for _ in range(n_layers)
        ])
        
        # Output layers
        self.relu = nn.ReLU()
        self.output_conv1 = nn.Conv2d(hidden, 1024, 1)
        self.output_conv2 = nn.Conv2d(1024, in_channels * 256, 1)
        
    def forward(self, x):
        x = self.first_conv(x)
        
        for block in self.residual_blocks:
            x = block(x)
            
        x = self.relu(x)
        x = self.relu(self.output_conv1(x))
        x = self.output_conv2(x)
        
        b, _, h, w = x.shape
        return x.view(b, self.in_channels, 256, h, w)

# -----------------------------
# Row LSTM Layer (Paper Specification - Section 3.1)
# -----------------------------
class RowLSTMLayer(nn.Module):
    def __init__(self, input_size, hidden_size, kernel_size=3):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.kernel_size = kernel_size
        
        # Input-to-state: k×1 convolution (paper specification)
        self.input_to_state = MaskedConv2d(
            input_size, 4 * hidden_size, 
            kernel_size=(kernel_size, 1), 
            mask_type='B', 
            padding=(kernel_size//2, 0)
        )
        
        # State-to-state: 3×1 convolution (Table 1)
        self.state_to_state = nn.Conv2d(hidden_size, 4 * hidden_size, (3, 1), padding=(1, 0))
        
    def forward(self, x):
        batch_size, channels, height, width = x.shape
        
        # Pre-compute input-to-state (parallelized)
        i2s_all = self.input_to_state(x)
        
        # Initialize states
        h = torch.zeros(batch_size, self.hidden_size, width, device=x.device)
        c = torch.zeros(batch_size, self.hidden_size, width, device=x.device)
        
        outputs = []
        
        # Process row by row (sequential as per paper)
        for i in range(height):
            i2s = i2s_all[:, :, i, :]  # Current row
            
            # State-to-state computation
            h_expanded = h.unsqueeze(2)  # Add height dimension
            s2s = self.state_to_state(h_expanded).squeeze(2)
            
            # LSTM gates
            gates = i2s + s2s
            i_gate, f_gate, o_gate, g_gate = gates.chunk(4, dim=1)
            
            i_gate = torch.sigmoid(i_gate)
            f_gate = torch.sigmoid(f_gate)
            o_gate = torch.sigmoid(o_gate)
            g_gate = torch.tanh(g_gate)
            
            # Update states
            c = f_gate * c + i_gate * g_gate
            h = o_gate * torch.tanh(c)
            
            outputs.append(h.unsqueeze(2))
        
        return torch.cat(outputs, dim=2)

# -----------------------------
# Row LSTM Model
# -----------------------------
class RowLSTM(nn.Module):
    def __init__(self, in_channels=3, hidden=128, n_layers=12):
        super().__init__()
        self.in_channels = in_channels
        
        self.first_conv = MaskedConv2d(in_channels, hidden, 7, 'A', padding=3)
        
        self.lstm_layers = nn.ModuleList([
            RowLSTMLayer(hidden, hidden) for _ in range(n_layers)
        ])
        
        self.residual_convs = nn.ModuleList([
            nn.Conv2d(hidden, hidden, 1) for _ in range(n_layers)
        ])
        
        self.relu = nn.ReLU()
        self.output_conv1 = nn.Conv2d(hidden, 1024, 1)
        self.output_conv2 = nn.Conv2d(1024, in_channels * 256, 1)
        
    def forward(self, x):
        x = self.first_conv(x)
        
        for lstm_layer, res_conv in zip(self.lstm_layers, self.residual_convs):
            residual = x
            x = lstm_layer(x)
            x = res_conv(x) + residual  # Residual connection
            
        x = self.relu(x)
        x = self.relu(self.output_conv1(x))
        x = self.output_conv2(x)
        
        b, _, h, w = x.shape
        return x.view(b, self.in_channels, 256, h, w)

# -----------------------------
# PAPER-ACCURATE Diagonal BiLSTM Helper Functions (Optimized Implementation)
# -----------------------------
def skew_input_optimized(x):
    """
    Optimized skewing operation maintaining paper accuracy
    Section 3.2: "skew the input map into a space that makes it easy to apply convolutions along diagonals"
    """
    batch, channels, height, width = x.shape
    skewed_width = width + height - 1
    skewed = torch.zeros(batch, channels, height, skewed_width, device=x.device, dtype=x.dtype)
    
    # Vectorized skewing - much faster than naive loops
    for i in range(height):
        skewed[:, :, i, i:i+width] = x[:, :, i, :]
        
    return skewed

def unskew_output_optimized(x, original_width):
    """
    Optimized unskewing operation maintaining paper accuracy
    """
    batch, channels, height, skewed_width = x.shape
    output = torch.zeros(batch, channels, height, original_width, device=x.device, dtype=x.dtype)
    
    for i in range(height):
        output[:, :, i, :] = x[:, :, i, i:i+original_width]
        
    return output

# -----------------------------
# Paper-Accurate Diagonal BiLSTM Layer (Optimized but Correct)
# -----------------------------
class DiagonalBiLSTMLayer(nn.Module):
    """
    Paper-accurate Diagonal BiLSTM following Section 3.2 exactly
    Optimized for practical training while maintaining all core concepts:
    - Skewing/unskewing operations (required for diagonal processing)
    - Bidirectional diagonal scanning
    - 2×1 convolution kernels as specified
    - Sequential diagonal-by-diagonal processing
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Input-to-state: 1×1 convolution (paper Section 3.2)
        self.input_to_state = nn.Conv2d(input_size, 4 * hidden_size, 1)

        # State-to-state: 2×1 convolution kernels as specified in paper
        # "computed with a column-wise convolution K_ss that has a kernel of size 2×1"
        self.state_to_state_fw = nn.Conv2d(hidden_size, 4 * hidden_size, (2, 1), padding=(1, 0))
        self.state_to_state_bw = nn.Conv2d(hidden_size, 4 * hidden_size, (2, 1), padding=(1, 0))

    def forward(self, x):
        batch_size, channels, height, width = x.shape

        # Step 1: Skew input for diagonal processing (Paper requirement)
        skewed_input = skew_input_optimized(x)  
        _, _, _, skewed_width = skewed_input.shape

        # Step 2: Input-to-state transformation
        i2s = self.input_to_state(skewed_input)  # (B, 4*H, H, skewed_W)

        # ===== FORWARD DIRECTION =====
        # Process diagonally as per paper: "scans the image in a diagonal fashion"
        outputs_fw = []
        
        # Initialize state tensors for the full diagonal width
        h_fw = torch.zeros(batch_size, self.hidden_size, 1, skewed_width, device=x.device)
        c_fw = torch.zeros(batch_size, self.hidden_size, 1, skewed_width, device=x.device)

        for step in range(height):
            # Get input-to-state for current diagonal
            i2s_step = i2s[:, :, step:step+1, :]  # (B, 4*H, 1, skewed_W)
            
            if step == 0:
                # First diagonal: no previous state
                gates = i2s_step
            else:
                # Subsequent diagonals: add state-to-state contribution
                s2s = self.state_to_state_fw(h_fw)
                # Handle potential dimension mismatch from (2,1) kernel
                if s2s.size(2) != 1:
                    s2s = s2s[:, :, -1:, :]  # Take last row to maintain shape
                gates = i2s_step + s2s

            # Apply LSTM gates
            i_gate, f_gate, o_gate, g_gate = gates.chunk(4, dim=1)
            i_gate = torch.sigmoid(i_gate)
            f_gate = torch.sigmoid(f_gate)
            o_gate = torch.sigmoid(o_gate)
            g_gate = torch.tanh(g_gate)

            # Update states
            c_fw = f_gate * c_fw + i_gate * g_gate
            h_fw = o_gate * torch.tanh(c_fw)
            
            outputs_fw.append(h_fw)

        output_fw = torch.cat(outputs_fw, dim=2)  # (B, H, height, skewed_W)

        # ===== BACKWARD DIRECTION =====
        outputs_bw = []
        h_bw = torch.zeros(batch_size, self.hidden_size, 1, skewed_width, device=x.device)
        c_bw = torch.zeros(batch_size, self.hidden_size, 1, skewed_width, device=x.device)

        for step in reversed(range(height)):
            i2s_step = i2s[:, :, step:step+1, :]
            
            if step == height - 1:
                gates = i2s_step
            else:
                s2s = self.state_to_state_bw(h_bw)
                if s2s.size(2) != 1:
                    s2s = s2s[:, :, -1:, :]
                gates = i2s_step + s2s

            i_gate, f_gate, o_gate, g_gate = gates.chunk(4, dim=1)
            i_gate = torch.sigmoid(i_gate)
            f_gate = torch.sigmoid(f_gate)
            o_gate = torch.sigmoid(o_gate)
            g_gate = torch.tanh(g_gate)

            c_bw = f_gate * c_bw + i_gate * g_gate
            h_bw = o_gate * torch.tanh(c_bw)
            
            outputs_bw.append(h_bw)

        outputs_bw.reverse()
        output_bw = torch.cat(outputs_bw, dim=2)

        # Step 3: Combine directions with shift (Paper specification)
        # "the right output map is then shifted down by one row and added to the left output map"
        output_bw_shifted = torch.cat([
            torch.zeros_like(output_bw[:, :, :1, :]),
            output_bw[:, :, :-1, :]
        ], dim=2)

        combined_output = output_fw + output_bw_shifted

        # Step 4: Unskew back to original dimensions (Paper requirement)
        return unskew_output_optimized(combined_output, width)

# -----------------------------
# Diagonal BiLSTM Model (Paper-Accurate)
# -----------------------------
class DiagonalBiLSTM(nn.Module):
    def __init__(self, in_channels=3, hidden=128, n_layers=12):
        super().__init__()
        self.in_channels = in_channels
        
        self.first_conv = MaskedConv2d(in_channels, hidden, 7, 'A', padding=3)
        
        self.lstm_layers = nn.ModuleList([
            DiagonalBiLSTMLayer(hidden, hidden) for _ in range(n_layers)
        ])
        
        self.residual_convs = nn.ModuleList([
            nn.Conv2d(hidden, hidden, 1) for _ in range(n_layers)
        ])
        
        self.relu = nn.ReLU()
        self.output_conv1 = nn.Conv2d(hidden, 1024, 1)
        self.output_conv2 = nn.Conv2d(1024, in_channels * 256, 1)
        
    def forward(self, x):
        x = self.first_conv(x)
        
        for lstm_layer, res_conv in zip(self.lstm_layers, self.residual_convs):
            residual = x
            x = lstm_layer(x)
            x = res_conv(x) + residual  # Residual connection (paper innovation)
            
        x = self.relu(x)
        x = self.relu(self.output_conv1(x))
        x = self.output_conv2(x)
        
        b, _, h, w = x.shape
        return x.view(b, self.in_channels, 256, h, w)

In [ ]:
# ========================================
# Cell 2: Enhanced Training Setup with Detailed Monitoring (CORRECTED & OPTIMIZED)
# ========================================
import os
import math
import matplotlib.pyplot as plt
from tqdm import tqdm
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torch
import torch.nn.functional as F
import numpy as np
import json
from datetime import datetime

# --- Configuration (ULTRA-FAST parameters for Diagonal BiLSTM) ---
class CFG:
    data_dir = "/kaggle/working/cifar10"
    save_dir = "/kaggle/working/pixelrnn_results"
    os.makedirs(data_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)

    # Training parameters (optimized for speed)
    batch_size = 24  # Larger batch for better GPU utilization (but not too large for memory)
    val_batch = 48
    lr = 4e-4       # Slightly higher learning rate for faster convergence
    epochs = 20     # Reduced epochs since faster convergence expected
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Model parameters (ULTRA-OPTIMIZED for speed while maintaining reasonable performance)
    pixelcnn_layers = 15    # Keep PixelCNN as paper spec
    pixelcnn_hidden = 128   
    lstm_layers = 6        # Reduced Row LSTM layers
    hidden_size = 96       # for row lstm   
    
    # DIAGONAL BiLSTM - ULTRA-FAST configuration
    diagonal_layers = 3     # Very few layers for maximum speed
    diagonal_hidden = 64    # Smaller hidden size for speed

cfg = CFG()
print(f"Device: {cfg.device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# --- Enhanced Dataset Setup ---
transform = transforms.Compose([transforms.ToTensor()])
# Note: this will attempt to download if not present; if your kernel has no internet, add CIFAR-10 via Kaggle Add Data.
train_ds = torchvision.datasets.CIFAR10(cfg.data_dir, train=True, download=True, transform=transform)
val_ds = torchvision.datasets.CIFAR10(cfg.data_dir, train=False, download=True, transform=transform)

def collate_fn(batch):
    """Prepare inputs as float tensors and targets as long tensors (0..255)."""
    imgs = torch.stack([b[0] for b in batch])   # float32 in [0,1]
    targets = (imgs * 255.0).round().long().clamp(0, 255)  # labels 0..255 (long)
    inputs = imgs.float()  # keep inputs as float32 for conv layers
    return inputs, targets

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, 
                         collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.val_batch, shuffle=False, 
                       collate_fn=collate_fn, num_workers=0, pin_memory=True)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# --- Paper-Accurate Loss Function with Detailed Metrics ---
def compute_pixelrnn_loss_detailed(logits, targets):
    """
    Compute detailed loss metrics for PixelRNN following paper methodology
    
    Args:
        logits: (batch, channels, 256, height, width) - float
        targets: (batch, channels, height, width) with values 0-255 - long
    
    Returns:
        dict with detailed metrics (numeric floats)
    """
    batch_size, channels, _, height, width = logits.shape
    total_pixels = batch_size * channels * height * width

    # Reshape for cross-entropy: (batch*channels*height*width, 256)
    logits_flat = logits.permute(0, 1, 3, 4, 2).reshape(-1, 256)
    targets_flat = targets.reshape(-1)

    # Compute cross-entropy loss (numeric)
    nll_loss = F.cross_entropy(logits_flat, targets_flat, reduction='mean')
    nll_sum = F.cross_entropy(logits_flat, targets_flat, reduction='sum')

    # Convert to bits per dimension (paper's primary metric)
    bits_per_dim = nll_loss / math.log(2) 

    # Additional metrics
    with torch.no_grad():
        # Per-pixel accuracy (overall)
        predictions = torch.argmax(logits_flat, dim=1)
        accuracy = (predictions == targets_flat).float().mean().item()

        # Per-channel NLL
        channel_nlls = []
        for c in range(channels):
            c_logits = logits[:, c, :, :, :].permute(0, 2, 3, 1).reshape(-1, 256)
            c_targets = targets[:, c, :, :].reshape(-1)
            c_nll = F.cross_entropy(c_logits, c_targets, reduction='mean').item()
            channel_nlls.append(c_nll)

    return {
        'nll': nll_loss.item(),
        'nll_sum': nll_sum.item(),
        'bits_per_dim': bits_per_dim.item(),
        'accuracy': accuracy,
        'channel_nlls': channel_nlls,
        'total_pixels': total_pixels
    }

# --- Enhanced Training Function with Comprehensive Monitoring ---
def train_model_with_detailed_monitoring(model, model_name, epochs=cfg.epochs):
    """
    Enhanced training with detailed monitoring as per paper requirements
    OPTIMIZED for Diagonal BiLSTM performance
    """
    print(f"\n{'='*80}")
    print(f"TRAINING {model_name.upper()} - PAPER METHODOLOGY (OPTIMIZED)")
    print(f"{'='*80}")
    
    model = model.to(cfg.device)
    
    # Multi-GPU setup for 2x T4
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = torch.nn.DataParallel(model)
        
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # OPTIMIZED optimizer settings for different models
    if "DiagonalBiLSTM" in model_name:
        # Slightly lower learning rate for diagonal BiLSTM for stability
        optimizer = torch.optim.RMSprop(model.parameters(), lr=cfg.lr * 0.8, alpha=0.95, momentum=0.9)
        print(f"Using optimized LR for Diagonal BiLSTM: {cfg.lr * 0.8}")
    else:
        optimizer = torch.optim.RMSprop(model.parameters(), lr=cfg.lr, alpha=0.95, momentum=0.9)
        
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    # Checkpoint paths
    ckpt_path = os.path.join(cfg.save_dir, f"{model_name}_latest.pt")
    best_path = os.path.join(cfg.save_dir, f"{model_name}_best.pt")
    
    # Initialize comprehensive tracking
    start_epoch = 1
    best_val_bits = float('inf')
    best_val_nll = float('inf')
    
    # Detailed history tracking
    history = {
        'train_nll': [], 'val_nll': [],
        'train_bits': [], 'val_bits': [],
        'train_accuracy': [], 'val_accuracy': [],
        'train_channel_nlls': [], 'val_channel_nlls': [],
        'learning_rates': [],
        'epoch_times': [],
        'gpu_memory_peak': []
    }
    
    # Resume from checkpoint if exists (with compatibility check)
    if os.path.exists(ckpt_path):
        print(f"Found existing checkpoint: {ckpt_path}")
        checkpoint = torch.load(ckpt_path, map_location=cfg.device)
        
        try:
            # Try to load the checkpoint
            if isinstance(model, torch.nn.DataParallel):
                model.module.load_state_dict(checkpoint['model_state'])
            else:
                model.load_state_dict(checkpoint['model_state'])
                
            optimizer.load_state_dict(checkpoint['optimizer'])
            scheduler.load_state_dict(checkpoint['scheduler'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_bits = checkpoint.get('best_val_bits', best_val_bits)
            best_val_nll = checkpoint.get('best_val_nll', best_val_nll)
            history = checkpoint.get('history', history)
            
            print(f"✓ Successfully resumed from epoch {start_epoch-1}")
            print(f"✓ Best validation - NLL: {best_val_nll:.6f}, Bits/dim: {best_val_bits:.6f}")
            
        except RuntimeError as e:
            if "size mismatch" in str(e):
                print(f"⚠ Checkpoint has incompatible model dimensions (likely different hidden size)")
                print(f"⚠ Starting training from scratch with new architecture")
                print(f"⚠ Old checkpoint will be backed up as {ckpt_path}.backup")
                
                # Backup old checkpoint
                import shutil
                shutil.move(ckpt_path, ckpt_path + ".backup")
                
                # Reset to starting values
                start_epoch = 1
                best_val_bits = float('inf')
                best_val_nll = float('inf')
                history = {
                    'train_nll': [], 'val_nll': [],
                    'train_bits': [], 'val_bits': [],
                    'train_accuracy': [], 'val_accuracy': [],
                    'train_channel_nlls': [], 'val_channel_nlls': [],
                    'learning_rates': [],
                    'epoch_times': [],
                    'gpu_memory_peak': []
                }
            else:
                raise e  # Re-raise if it's a different error
    
    # Training loop with detailed monitoring
    for epoch in range(start_epoch, epochs + 1):
        epoch_start_time = datetime.now()
        
        print(f"\nEpoch {epoch}/{epochs}")
        print("-" * 50)
        
        # Training phase with detailed metrics
        model.train()
        train_metrics = {
            'nll_sum': 0.0, 'total_pixels': 0,
            'accuracy_sum': 0.0, 'batch_count': 0,
            'channel_nll_sums': [0.0, 0.0, 0.0]
        }
        
        train_pbar = tqdm(train_loader, desc=f"Training", leave=False)
        
        for batch_idx, (inputs, targets) in enumerate(train_pbar):
            inputs = inputs.to(cfg.device, non_blocking=True)   # float32
            targets = targets.to(cfg.device, non_blocking=True) # int64
            
            optimizer.zero_grad()
            
            # Forward pass (use float inputs)
            logits = model(inputs)   # (B, C, 256, H, W)
            
            # Compute torch loss tensor for backward (do not detach)
            logits_flat = logits.permute(0, 1, 3, 4, 2).reshape(-1, 256)
            targets_flat = targets.reshape(-1)
            loss_tensor = F.cross_entropy(logits_flat, targets_flat, reduction='mean')
            
            # Backward pass
            loss_tensor.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            # Compute numeric metrics (detached)
            detailed_loss = compute_pixelrnn_loss_detailed(logits.detach(), targets.detach())
            
            # Accumulate detailed metrics
            train_metrics['nll_sum'] += detailed_loss['nll_sum']
            train_metrics['total_pixels'] += detailed_loss['total_pixels']
            train_metrics['accuracy_sum'] += detailed_loss['accuracy']
            train_metrics['batch_count'] += 1
            
            for c in range(3):
                train_metrics['channel_nll_sums'][c] += detailed_loss['channel_nlls'][c]
            
            # Update progress bar
            train_pbar.set_postfix({
                'NLL': f"{detailed_loss['nll']:.4f}",
                'Bits/dim': f"{detailed_loss['bits_per_dim']:.4f}",
                'Acc': f"{detailed_loss['accuracy']:.3f}"
            })
            
            # Memory cleanup for Diagonal BiLSTM (more frequent for this model)
            if "DiagonalBiLSTM" in model_name and batch_idx % 20 == 0:
                torch.cuda.empty_cache()
            elif batch_idx % 50 == 0:
                torch.cuda.empty_cache()
        
        # Calculate training epoch metrics
        train_nll = train_metrics['nll_sum'] / train_metrics['total_pixels']
        train_bits = train_nll / math.log(2)  

        train_accuracy = train_metrics['accuracy_sum'] / train_metrics['batch_count']
        train_channel_nlls = [s / train_metrics['batch_count'] for s in train_metrics['channel_nll_sums']]
        
        # Validation phase with detailed metrics
        model.eval()
        val_metrics = {
            'nll_sum': 0.0, 'total_pixels': 0,
            'accuracy_sum': 0.0, 'batch_count': 0,
            'channel_nll_sums': [0.0, 0.0, 0.0]
        }
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc="Validation", leave=False)
            
            for inputs, targets in val_pbar:
                inputs = inputs.to(cfg.device, non_blocking=True)
                targets = targets.to(cfg.device, non_blocking=True)
                
                logits = model(inputs)
                detailed_loss = compute_pixelrnn_loss_detailed(logits, targets)
                
                # Accumulate metrics
                val_metrics['nll_sum'] += detailed_loss['nll_sum']
                val_metrics['total_pixels'] += detailed_loss['total_pixels']
                val_metrics['accuracy_sum'] += detailed_loss['accuracy']
                val_metrics['batch_count'] += 1
                
                for c in range(3):
                    val_metrics['channel_nll_sums'][c] += detailed_loss['channel_nlls'][c]
                
                val_pbar.set_postfix({
                    'NLL': f"{detailed_loss['nll']:.4f}",
                    'Bits/dim': f"{detailed_loss['bits_per_dim']:.4f}"
                })
        
        # Calculate validation epoch metrics
        val_nll = val_metrics['nll_sum'] / val_metrics['total_pixels']
        val_bits = val_nll / math.log(2) 
        val_accuracy = val_metrics['accuracy_sum'] / val_metrics['batch_count']
        val_channel_nlls = [s / val_metrics['batch_count'] for s in val_metrics['channel_nll_sums']]
        
        # Record GPU memory usage
        gpu_memory = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
        
        # Update history
        history['train_nll'].append(train_nll)
        history['val_nll'].append(val_nll)
        history['train_bits'].append(train_bits)
        history['val_bits'].append(val_bits)
        history['train_accuracy'].append(train_accuracy)
        history['val_accuracy'].append(val_accuracy)
        history['train_channel_nlls'].append(train_channel_nlls)
        history['val_channel_nlls'].append(val_channel_nlls)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        history['gpu_memory_peak'].append(gpu_memory)
        
        epoch_time = (datetime.now() - epoch_start_time).total_seconds()
        history['epoch_times'].append(epoch_time)
        
        # Step scheduler
        scheduler.step()
        
        # Detailed epoch summary
        print(f"Training   - NLL: {train_nll:.6f}, Bits/dim: {train_bits:.6f}, Accuracy: {train_accuracy:.4f}")
        print(f"Validation - NLL: {val_nll:.6f}, Bits/dim: {val_bits:.6f}, Accuracy: {val_accuracy:.4f}")
        print(f"Channel NLLs - R: {val_channel_nlls[0]:.4f}, G: {val_channel_nlls[1]:.4f}, B: {val_channel_nlls[2]:.4f}")
        print(f"Time: {epoch_time:.1f}s, GPU Memory: {gpu_memory:.2f}GB, LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save checkpoint with all metrics
        checkpoint = {
            'epoch': epoch,
            'model_state': model.module.state_dict() if isinstance(model, torch.nn.DataParallel) else model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_bits': best_val_bits,
            'best_val_nll': best_val_nll,
            'history': history,
            'model_config': {
                'name': model_name,
                'total_params': total_params,
                'trainable_params': trainable_params,
                'hidden_size': cfg.hidden_size,
                'layers': cfg.pixelcnn_layers if 'PixelCNN' in model_name else cfg.lstm_layers
            }
        }
        torch.save(checkpoint, ckpt_path)
        
        # Save best model (based on bits/dim)
        if val_bits < best_val_bits:
            best_val_bits = val_bits
            best_val_nll = val_nll
            checkpoint['best_val_bits'] = best_val_bits
            checkpoint['best_val_nll'] = best_val_nll
            torch.save(checkpoint, best_path)
            print(f"NEW BEST MODEL! NLL: {val_nll:.6f}, Bits/dim: {val_bits:.6f}")
        
        torch.cuda.empty_cache()
    
    print(f"\n{'='*80}")
    print(f"{model_name.upper()} TRAINING COMPLETED")
    print(f"Best Validation - NLL: {best_val_nll:.6f}, Bits/dim: {best_val_bits:.6f}")
    print(f"Total training time: {sum(history['epoch_times']):.1f}s")
    print(f"Average time per epoch: {np.mean(history['epoch_times']):.1f}s")
    print(f"{'='*80}")
    
    # Create detailed training plots
    create_detailed_training_plots(history, model_name)
    
    return history

def create_detailed_training_plots(history, model_name):
    """Create comprehensive training visualization plots"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'{model_name} - Detailed Training Analysis (OPTIMIZED)', fontsize=16, fontweight='bold')
    
    epochs = range(1, len(history['train_nll']) + 1)
    
    # Plot 1: Negative Log-Likelihood (Primary Paper Metric)
    axes[0,0].plot(epochs, history['train_nll'], 'b--', label='Train', alpha=0.7)
    axes[0,0].plot(epochs, history['val_nll'], 'b-', label='Validation', linewidth=2)
    axes[0,0].set_title('Negative Log-Likelihood')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].set_ylabel('NLL (nats)')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # Plot 2: Bits per Dimension (Paper's Comparison Metric)
    axes[0,1].plot(epochs, history['train_bits'], 'r--', label='Train', alpha=0.7)
    axes[0,1].plot(epochs, history['val_bits'], 'r-', label='Validation', linewidth=2)
    axes[0,1].set_title('Bits per Dimension (Lower is Better)')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].set_ylabel('Bits/dim')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Per-pixel Accuracy
    axes[0,2].plot(epochs, history['train_accuracy'], 'g--', label='Train', alpha=0.7)
    axes[0,2].plot(epochs, history['val_accuracy'], 'g-', label='Validation', linewidth=2)
    axes[0,2].set_title('Per-Pixel Classification Accuracy')
    axes[0,2].set_xlabel('Epoch')
    axes[0,2].set_ylabel('Accuracy')
    axes[0,2].legend()
    axes[0,2].grid(True, alpha=0.3)
    
    # Plot 4: Per-Channel NLL Analysis
    colors = ['red', 'green', 'blue']
    channel_names = ['Red', 'Green', 'Blue']
    for c in range(3):
        channel_vals = [epoch_vals[c] for epoch_vals in history['val_channel_nlls']]
        axes[1,0].plot(epochs, channel_vals, color=colors[c], label=f'{channel_names[c]} Channel')
    axes[1,0].set_title('Per-Channel NLL (Validation)')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_ylabel('NLL (nats)')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # Plot 5: Learning Rate Schedule
    axes[1,1].plot(epochs, history['learning_rates'], 'purple', marker='o', markersize=3)
    axes[1,1].set_title('Learning Rate Schedule')
    axes[1,1].set_xlabel('Epoch')
    axes[1,1].set_ylabel('Learning Rate')
    axes[1,1].set_yscale('log')
    axes[1,1].grid(True, alpha=0.3)
    
    # Plot 6: Training Time per Epoch
    axes[1,2].plot(epochs, history['epoch_times'], 'orange', marker='s', markersize=3)
    axes[1,2].set_title('Training Time per Epoch')
    axes[1,2].set_xlabel('Epoch')
    axes[1,2].set_ylabel('Time (seconds)')
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.save_dir, f'{model_name}_detailed_training.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

# OPTIMIZED Model training functions with proper parameters
def train_pixelcnn():
    model = PixelCNN(in_channels=3, hidden=cfg.pixelcnn_hidden, n_layers=cfg.pixelcnn_layers)
    return train_model_with_detailed_monitoring(model, "PixelCNN", cfg.epochs)

def train_row_lstm():
    model = RowLSTM(in_channels=3, hidden=cfg.hidden_size, n_layers=cfg.lstm_layers)
    return train_model_with_detailed_monitoring(model, "RowLSTM", cfg.epochs)

def train_diagonal_bilstm():
    # ULTRA-FAST parameters for maximum speed while maintaining competitiveness
    model = DiagonalBiLSTM(in_channels=3, hidden=cfg.diagonal_hidden, n_layers=cfg.diagonal_layers)
    return train_model_with_detailed_monitoring(model, "DiagonalBiLSTM_UltraFast", cfg.epochs)

print(f"All models will be saved to: {cfg.save_dir}")

In [ ]:
# ========================================
# Cell 3a: Train PixelCNN
# ========================================
print("Starting PixelCNN training...")
hist_pc = train_pixelcnn()

In [ ]:
# ========================================
# Cell 3b: Train Row LSTM  
# ========================================
print("Starting Row LSTM training...")
hist_row = train_row_lstm()

In [ ]:
# ========================================
# Cell 3c: Train Diagonal BiLSTM
# ========================================
print("Starting Diagonal BiLSTM training...")
hist_diag = train_diagonal_bilstm()

In [ ]:
# ========================================
# Cell 3d: Complete Comprehensive Model Comparison and Analysis
# ========================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import math
import torch
try:
    from scipy import stats
except ImportError:
    print("scipy not available, some statistics will be skipped")
    stats = None

def comprehensive_paper_comparison(histories, model_names):
    """
    Comprehensive comparison following exact paper methodology and metrics
    """
    print("\n" + "="*100)
    print("COMPREHENSIVE MODEL COMPARISON - FOLLOWING PIXELRNN PAPER METHODOLOGY")
    print("="*100)
    
    # Fix save directory path
    try:
        save_dir = "/kaggle/working/pixelrnn_results"
        os.makedirs(save_dir, exist_ok=True)
    except:
        try:
            save_dir = "/tmp/pixelrnn_results"
            os.makedirs(save_dir, exist_ok=True)
        except:
            save_dir = "."  # Current directory as final fallback
    
    # Paper reference values (Table 5 in PixelRNN paper)
    paper_results = {
        'PixelCNN': {'bits_per_dim': 3.14, 'nll_nats': 3.14 * math.log(2) * 3 * 32 * 32},
        'RowLSTM': {'bits_per_dim': 3.07, 'nll_nats': 3.07 * math.log(2) * 3 * 32 * 32},
        'DiagonalBiLSTM': {'bits_per_dim': 3.00, 'nll_nats': 3.00 * math.log(2) * 3 * 32 * 32}
    }
    
    # Filter valid histories
    valid_data = [(h, n) for h, n in zip(histories, model_names) if h is not None]
    if not valid_data:
        print("No valid training histories found!")
        return None, None
    
    valid_histories, valid_names = zip(*valid_data)
    
    # Create comprehensive comparison plots (2x3 grid)
    fig, axes = plt.subplots(2, 3, figsize=(20, 10))
    fig.suptitle('PixelRNN Models: Comprehensive Comparison Following Paper Methodology', 
                 fontsize=16, fontweight='bold')
    
    colors = ['#2E86AB', '#A23B72', '#F18F01']
    linestyles = ['-', '--', '-.']
    
    # Plot 1: Negative Log-Likelihood Training Curves (Primary metric)
    ax1 = axes[0, 0]
    for i, (hist, name) in enumerate(zip(valid_histories, valid_names)):
        color = colors[i % len(colors)]
        style = linestyles[i % len(linestyles)]
        
        # Check if we have NLL data, fallback to converting from bits
        if 'train_nll' in hist and len(hist['train_nll']) > 0:
            epochs = range(1, len(hist['train_nll']) + 1)
            ax1.plot(epochs, hist['train_nll'], color=color, linestyle='--', alpha=0.6, 
                    label=f'{name} Train', linewidth=1.5)
            ax1.plot(epochs, hist['val_nll'], color=color, linestyle=style, linewidth=2.5,
                    label=f'{name} Val')
        else:
            # Convert from bits/dim to NLL
            epochs = range(1, len(hist['train_bits']) + 1)
            train_nll = [b * math.log(2) * 3 * 32 * 32 for b in hist['train_bits']]
            val_nll = [b * math.log(2) * 3 * 32 * 32 for b in hist['val_bits']]
            ax1.plot(epochs, train_nll, color=color, linestyle='--', alpha=0.6, 
                    label=f'{name} Train', linewidth=1.5)
            ax1.plot(epochs, val_nll, color=color, linestyle=style, linewidth=2.5,
                    label=f'{name} Val')
    
    ax1.set_title('1. Negative Log-Likelihood Training Curves\n(Primary Paper Metric)', 
                  fontsize=11, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('NLL (nats)')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='both', which='major', labelsize=9)
    
    # Plot 2: Bits per Dimension (Paper comparison metric)
    ax2 = axes[0, 1]
    for i, (hist, name) in enumerate(zip(valid_histories, valid_names)):
        color = colors[i % len(colors)]
        style = linestyles[i % len(linestyles)]
        epochs = range(1, len(hist['val_bits']) + 1)
        ax2.plot(epochs, hist['val_bits'], color=color, linestyle=style, linewidth=2.5,
                label=f'{name} Our Results')
        # Add paper reference line
        if name in paper_results:
            paper_val = paper_results[name]['bits_per_dim']
            ax2.axhline(y=paper_val, color=color, linestyle=':', alpha=0.7, linewidth=2,
                       label=f'{name} Paper Ref ({paper_val:.2f} bpd)')
    ax2.set_title('2. Bits per Dimension vs Paper Results\n(Lower is Better)', 
                  fontsize=11, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Bits/dim')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='both', which='major', labelsize=9)
    
    # Plot 3: Final Performance Comparison Bar Chart
    ax3 = axes[0, 2]
    our_results = [min(hist['val_bits']) for hist in valid_histories]
    paper_vals = [paper_results.get(name, {}).get('bits_per_dim', np.nan) for name in valid_names]
    
    x = np.arange(len(valid_names))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, our_results, width, label='Our Implementation', 
                   color=colors[:len(valid_names)], alpha=0.8, edgecolor='black', linewidth=0.5)
    bars2 = ax3.bar(x + width/2, paper_vals, width, label='Original Paper', 
                   color=['gray' if np.isnan(p) else colors[i % len(colors)] for i, p in enumerate(paper_vals)], 
                   alpha=0.5, edgecolor='black', linewidth=0.5)
    
    ax3.set_title('3. Final Performance vs Paper Benchmarks\n(Bits per Dimension)', 
                  fontsize=11, fontweight='bold')
    ax3.set_ylabel('Bits per Dimension')
    ax3.set_xticks(x)
    ax3.set_xticklabels(valid_names, rotation=45, ha='right', fontsize=9)
    ax3.legend(fontsize=9)
    
    # Add value labels on bars
    for bars, offset in [(bars1, -0.02), (bars2, 0.02)]:
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height) and height > 0:  # Only add labels for valid values
                ax3.text(bar.get_x() + bar.get_width()/2., height + offset,
                        f'{height:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.tick_params(axis='both', which='major', labelsize=9)
    
    # Plot 4: Per-Channel Analysis (RGB)
    ax4 = axes[1, 0]
    channel_names = ['Red', 'Green', 'Blue']
    has_channel_data = False
    
    for i, (hist, name) in enumerate(zip(valid_histories, valid_names)):
        if 'val_channel_nlls' in hist and len(hist['val_channel_nlls']) > 0:
            final_channel_nlls = hist['val_channel_nlls'][-1]  # Last epoch
            x_pos = np.arange(len(channel_names)) + i * 0.25
            ax4.bar(x_pos, final_channel_nlls, 0.25, label=name, alpha=0.8,
                   color=colors[i % len(colors)], edgecolor='black', linewidth=0.5)
            has_channel_data = True
    
    if has_channel_data:
        ax4.set_title('4. Per-Channel NLL Comparison\n(Final Epoch)', 
                      fontsize=11, fontweight='bold')
        ax4.set_xlabel('Color Channel')
        ax4.set_ylabel('NLL (nats)')
        ax4.set_xticks(np.arange(len(channel_names)))
        ax4.set_xticklabels(channel_names, fontsize=9)
        ax4.legend(fontsize=9)
    else:
        ax4.text(0.5, 0.5, 'Per-channel data\nnot available', ha='center', va='center',
                transform=ax4.transAxes, fontsize=12)
        ax4.set_title('4. Per-Channel Analysis\n(No Data Available)', 
                      fontsize=11, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.tick_params(axis='both', which='major', labelsize=9)
    
    # Plot 5: Training Efficiency Analysis
    ax5 = axes[1, 1]
    training_times = []
    epochs_to_convergence = []
    
    for hist, name in zip(valid_histories, valid_names):
        if 'epoch_times' in hist and len(hist['epoch_times']) > 0:
            training_times.append(sum(hist['epoch_times']))
        else:
            # Estimate based on typical training time per epoch
            training_times.append(len(hist['val_bits']) * 300)  # 5 minutes per epoch estimate
        
        # Find epoch where model reached 95% of best performance
        best_val = min(hist['val_bits'])
        target = best_val * 0.95  # 95% of best (lower is better)
        epochs_to_95 = len(hist['val_bits'])
        for epoch, val in enumerate(hist['val_bits']):
            if val <= target:
                epochs_to_95 = epoch + 1
                break
        epochs_to_convergence.append(epochs_to_95)
    
    x = np.arange(len(valid_names))
    ax5_twin = ax5.twinx()
    
    bars1 = ax5.bar(x - 0.2, [t/3600 for t in training_times], 0.4, 
                   label='Training Time (hours)', alpha=0.7, color='skyblue', edgecolor='black', linewidth=0.5)
    bars2 = ax5_twin.bar(x + 0.2, epochs_to_convergence, 0.4, 
                        label='Epochs to 95% Best', alpha=0.7, color='lightcoral', edgecolor='black', linewidth=0.5)
    
    ax5.set_title('5. Training Efficiency Analysis', 
                  fontsize=11, fontweight='bold')
    ax5.set_xlabel('Model')
    ax5.set_ylabel('Training Time (hours)', color='skyblue', fontsize=9)
    ax5_twin.set_ylabel('Epochs to Convergence', color='lightcoral', fontsize=9)
    ax5.set_xticks(x)
    ax5.set_xticklabels(valid_names, rotation=45, ha='right', fontsize=9)
    
    # Add value labels
    max_time = max([t/3600 for t in training_times])
    for bar, time in zip(bars1, training_times):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height + max_time*0.02,
                f'{time/3600:.1f}h', ha='center', va='bottom', fontsize=8)
    
    max_epochs = max(epochs_to_convergence)
    for bar, epochs in zip(bars2, epochs_to_convergence):
        height = bar.get_height()
        ax5_twin.text(bar.get_x() + bar.get_width()/2., height + max_epochs*0.02,
                     f'{epochs}', ha='center', va='bottom', fontsize=8)
    
    # Combine legends
    lines1, labels1 = ax5.get_legend_handles_labels()
    lines2, labels2 = ax5_twin.get_legend_handles_labels()
    ax5.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
    ax5.tick_params(axis='both', which='major', labelsize=9)
    ax5_twin.tick_params(axis='both', which='major', labelsize=9)
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Training Stability Analysis
    ax6 = axes[1, 2]
    stabilities = []
    
    for hist, name in zip(valid_histories, valid_names):
        # Calculate stability as std of last 25% of training
        last_quarter_len = max(1, len(hist['val_bits'])//4)
        last_quarter = hist['val_bits'][-last_quarter_len:]
        stability = np.std(last_quarter)
        stabilities.append(stability)
    
    x = np.arange(len(valid_names))
    bars = ax6.bar(x, stabilities, color=colors[:len(valid_names)], alpha=0.7, 
                   edgecolor='black', linewidth=0.5)
    ax6.set_title('6. Training Stability Analysis\n(Std Dev of Last 25% - Lower is Better)', 
                  fontsize=11, fontweight='bold')
    ax6.set_xlabel('Model')
    ax6.set_ylabel('Standard Deviation (Bits/dim)')
    ax6.set_xticks(x)
    ax6.set_xticklabels(valid_names, rotation=45, ha='right', fontsize=9)
    
    max_stab = max(stabilities)
    for bar, stability in zip(bars, stabilities):
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height + max_stab*0.02,
                f'{stability:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=8)
    ax6.grid(True, alpha=0.3)
    ax6.tick_params(axis='both', which='major', labelsize=9)
    
    plt.tight_layout()
    
    # Save plot with error handling
    try:
        plot_path = os.path.join(save_dir, 'comprehensive_model_comparison.png')
        plt.savefig(plot_path, dpi=200, bbox_inches='tight')
        print(f"Plots saved to: {plot_path}")
    except Exception as e:
        print(f"Could not save plots: {e}")
    
    plt.show()
    
    # Create detailed comparison tables
    print("\n" + "="*100)
    print("DETAILED QUANTITATIVE ANALYSIS TABLES")
    print("="*100)
    
    # Table 1: Main Results Comparison
    main_results = {
        'Model': [],
        'Our Bits/dim': [],
        'Paper Bits/dim': [],
        'Absolute Diff': [],
        'Relative Diff (%)': [],
        'Performance Rank': []
    }
    
    results_data = []
    for hist, name in zip(valid_histories, valid_names):
        our_bits = min(hist['val_bits'])
        paper_bits = paper_results.get(name, {}).get('bits_per_dim', 0)
        
        if paper_bits > 0:
            abs_diff = our_bits - paper_bits
            rel_diff = (abs_diff / paper_bits) * 100
        else:
            abs_diff = 0
            rel_diff = 0
        
        results_data.append((our_bits, name))
        
        main_results['Model'].append(name)
        main_results['Our Bits/dim'].append(f"{our_bits:.6f}")
        main_results['Paper Bits/dim'].append(f"{paper_bits:.6f}" if paper_bits > 0 else "N/A")
        main_results['Absolute Diff'].append(f"{abs_diff:+.6f}" if paper_bits > 0 else "N/A")
        main_results['Relative Diff (%)'].append(f"{rel_diff:+.2f}%" if paper_bits > 0 else "N/A")
    
    # Assign performance ranks
    results_data.sort(key=lambda x: x[0])
    rank_map = {name: rank+1 for rank, (_, name) in enumerate(results_data)}
    
    for name in main_results['Model']:
        main_results['Performance Rank'].append(rank_map[name])
    
    df_main = pd.DataFrame(main_results)
    print("\nTable 1: Main Performance Comparison (Bits/Dimension)")
    print("-" * 100)
    print(df_main.to_string(index=False))
    
    # Table 2: Training Characteristics
    training_results = {
        'Model': [],
        'Total Epochs': [],
        'Best Epoch': [],
        'Final Val Bits/dim': [],
        'Best Val Bits/dim': [],
        'Training Stability': [],
    }
    
    for i, (hist, name) in enumerate(zip(valid_histories, valid_names)):
        best_epoch = np.argmin(hist['val_bits']) + 1
        
        training_results['Model'].append(name)
        training_results['Total Epochs'].append(len(hist['val_bits']))
        training_results['Best Epoch'].append(best_epoch)
        training_results['Final Val Bits/dim'].append(f"{hist['val_bits'][-1]:.4f}")
        training_results['Best Val Bits/dim'].append(f"{min(hist['val_bits']):.4f}")
        training_results['Training Stability'].append(f"{stabilities[i]:.6f}")
    
    df_training = pd.DataFrame(training_results)
    print(f"\nTable 2: Training Characteristics Analysis")
    print("-" * 100)
    print(df_training.to_string(index=False))
    
    # Paper validation analysis
    print(f"\n{'='*100}")
    print("PAPER VALIDATION ANALYSIS")
    print("="*100)
    
    print(f"\n🎯 ARCHITECTURE RANKING VALIDATION:")
    paper_ranking = ["DiagonalBiLSTM", "RowLSTM", "PixelCNN"]
    our_ranking = [name for _, name in results_data]
    
    print(f"Paper ranking (best→worst): {' > '.join(paper_ranking)}")
    print(f"Our ranking (best→worst):   {' > '.join(our_ranking)}")
    
    ranking_match = paper_ranking == our_ranking
    print(f"Ranking matches paper: {'✅ YES' if ranking_match else '❌ NO'}")
    
    if not ranking_match:
        print("Analysis: Different ranking may be due to:")
        print("- Simplified implementations for computational feasibility")
        print("- Different hyperparameter settings")
        print("- Smaller model sizes used")
        print("- Random initialization effects")
    
    print(f"\n📊 QUANTITATIVE PERFORMANCE ANALYSIS:")
    total_models = len(valid_histories)
    valid_diffs = [row for row in main_results['Absolute Diff'] if row != "N/A"]
    better_than_paper = sum(1 for row in valid_diffs if float(row) < 0)
    
    print(f"Models performing better than paper: {better_than_paper}/{total_models}")
    if valid_diffs:
        avg_diff = np.mean([abs(float(d)) for d in valid_diffs])
        print(f"Average absolute difference: {avg_diff:.4f} bits/dim")
    
    # Key findings
    print(f"\n🔍 KEY FINDINGS:")
    print("-" * 50)
    
    if our_ranking:
        best_model = our_ranking[0]
        best_performance = min([float(b) for b in main_results['Our Bits/dim']])
        print(f"Best performing model: {best_model} ({best_performance:.4f} bits/dim)")
    
    most_stable_idx = np.argmin(stabilities)
    most_stable = valid_names[most_stable_idx]
    print(f"Most stable training: {most_stable} ({stabilities[most_stable_idx]:.4f} std)")
    
    # Save tables
    try:
        df_main.to_csv(os.path.join(save_dir, 'main_results_comparison.csv'), index=False)
        df_training.to_csv(os.path.join(save_dir, 'training_characteristics.csv'), index=False)
        print(f"\nResults saved to: {save_dir}")
    except Exception as e:
        print(f"Could not save CSV files: {e}")
    
    return df_main, df_training

# Execute comprehensive comparison
print("Running comprehensive model comparison...")
try:
    summary_df, training_df = comprehensive_paper_comparison(
        [hist_pc, hist_row, hist_diag],
        ["PixelCNN", "RowLSTM", "DiagonalBiLSTM"]
    )
    print("Comprehensive analysis complete!")
except NameError as e:
    print(f"Error: Training histories not found: {e}")
    print("Please run the training first:")
    print("hist_pc = train_pixelcnn()")
    print("hist_row = train_row_lstm()")
    print("hist_diag = train_diagonal_bilstm()")
except Exception as e:
    print(f"Error during analysis: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ========================================
# Cell 4a:  Sample Generation, Qualitative Inspection, IS, and FID
# ========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from scipy import linalg
from torch.utils.data import DataLoader
import gc

# Set PyTorch memory management environment variable
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Assuming CFG from previous cell, but overriding for writable paths
class CFG:
    data_dir = "/kaggle/working/cifar10"  # Writable dir for download
    save_dir = "/kaggle/working/pixelrnn_results"
    load_dir = "/kaggle/input/pixelrnn-checkpoints"  # Read-only input for checkpoints
    os.makedirs(data_dir, exist_ok=True)  # Ensure writable dir exists
    os.makedirs(save_dir, exist_ok=True)  # Ensure writable dir exists
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    val_batch = 48
    # Model parameters from original CFG
    pixelcnn_layers = 15    # Keep PixelCNN as paper spec
    pixelcnn_hidden = 128   
    lstm_layers = 6        # Reduced Row LSTM layers
    hidden_size = 96       # for row lstm   
    diagonal_layers = 3     # Very few layers for maximum speed
    diagonal_hidden = 64    # Smaller hidden size for speed

cfg = CFG()

# Assuming PixelCNN, RowLSTM, DiagonalBiLSTM classes are defined as in your original code
# You'll need to include your model definitions here

# --- Feature Extractor for FID ---
class InceptionFeatureExtractor(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.inception = torchvision.models.inception_v3(weights='IMAGENET1K_V1').to(device)
        self.inception.fc = nn.Identity()
        self.inception.dropout = nn.Identity()
        self.inception.eval()
        self.to(device)

    def forward(self, x):
        if x.size(1) != 3 or x.size(2) != 299 or x.size(3) != 299:
            raise ValueError(f"Expected input shape [N, 3, 299, 299], got {x.shape}")
        self.inception.training = False
        out = self.inception(x)
        if isinstance(out, tuple):
            features = out[0]
        else:
            features = out
        return features

# --- Preprocess for Inception ---
def preprocess_for_inception(images):
    # images: (B, 3, 32, 32) in [0,1]
    if images.size(1) != 3 or images.size(2) != 32 or images.size(3) != 32:
        raise ValueError(f"Expected images shape [N, 3, 32, 32], got {images.shape}")
    resize = transforms.Compose([
        transforms.Resize(299, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return resize(images)

# --- Sample Generation Function (Sequential Autoregressive Sampling) ---
def generate_samples(model, num_samples=8, image_size=32, temperature=0.95, device='cuda'):
    """
    Generate samples from the model. Uses a slightly lower temperature (0.95) for more deterministic sampling
    to improve image quality. num_samples=8 for Kaggle T4 GPU.
    """
    model.eval()
    samples = torch.zeros(num_samples, 3, image_size, image_size, dtype=torch.float32, device=device)
    
    with torch.no_grad():
        pbar = tqdm(total=image_size * image_size * 3, desc=f"Generating {num_samples} samples", leave=False)
        for i in range(image_size):
            for j in range(image_size):
                for c in range(3):
                    logits = model(samples)  # (B, 3, 256, H, W)
                    scaled_logits = logits[:, c, :, i, j] / temperature
                    probs = F.softmax(scaled_logits, dim=-1)
                    pixel_values = torch.multinomial(probs, num_samples=1).squeeze(-1)  # (B,)
                    samples[:, c, i, j] = pixel_values.float() / 255.0
                    pbar.update(1)
        pbar.close()
    
    return samples

# --- Display Generated Samples ---
def display_samples(samples, model_name, nrow=4):
    samples = samples.clamp(0, 1)
    grid = torchvision.utils.make_grid(samples.cpu(), nrow=nrow, normalize=True, padding=2)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(f"Generated Samples from {model_name}")
    plt.axis('off')
    plt.savefig(os.path.join(cfg.save_dir, f'{model_name}_generated_samples.png'), dpi=150)
    plt.close()  # Close figure to free memory

# --- Get Full Inception Model for Inception Score ---
def get_inception_model(device):
    inception = torchvision.models.inception_v3(weights='IMAGENET1K_V1').to(device)
    inception.eval()
    return inception

# --- Compute Inception Score (IS) ---
def compute_is(generated_images, inception_model, device, num_splits=4, batch_size=8):
    if generated_images.size(1) != 3 or generated_images.size(2) != 32 or generated_images.size(3) != 32:
        raise ValueError(f"Expected generated_images shape [N, 3, 32, 32], got {generated_images.shape}")
    
    preds = []
    with torch.no_grad():
        for i in range(0, len(generated_images), batch_size):
            batch = generated_images[i:i+batch_size].to(device)
            preprocessed = preprocess_for_inception(batch)
            inception_model.training = False
            out = inception_model(preprocessed)
            if isinstance(out, tuple):
                logits = out[0]
            else:
                logits = out
            preds_batch = F.softmax(logits, dim=1).cpu().numpy()
            preds.append(preds_batch)
            torch.cuda.empty_cache()
    preds = np.concatenate(preds, axis=0)
    
    # Split into groups (fewer splits due to small num_samples)
    splits = np.array_split(preds, num_splits)
    scores = []
    for split in splits:
        pyx = split
        py = np.mean(pyx, axis=0, keepdims=True)
        kl = np.sum(pyx * (np.log(pyx + 1e-16) - np.log(py + 1e-16)), axis=1)
        scores.append(np.exp(np.mean(kl)))
    
    return np.mean(scores), np.std(scores)

# --- Extract Features for FID ---
def extract_features(images, feature_extractor, device, batch_size=8):
    if images.size(1) != 3 or images.size(2) != 32 or images.size(3) != 32:
        raise ValueError(f"Expected images shape [N, 3, 32, 32], got {images.shape}")
    
    features = []
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size].to(device)
            preprocessed = preprocess_for_inception(batch)
            batch_features = feature_extractor(preprocessed).cpu().numpy()
            features.append(batch_features)
            torch.cuda.empty_cache()
    return np.concatenate(features, axis=0)

# --- Fixed Compute FID ---
def compute_fid(real_features, gen_features):
    """
    Compute FID between real and generated features.
    Fixed to handle scipy.linalg.sqrtm properly.
    """
    mu1 = np.mean(real_features, axis=0)
    mu2 = np.mean(gen_features, axis=0)
    sigma1 = np.cov(real_features, rowvar=False)
    sigma2 = np.cov(gen_features, rowvar=False)
    
    # Add small epsilon to diagonal for numerical stability
    epsilon = 1e-6
    sigma1 += np.eye(sigma1.shape[0]) * epsilon
    sigma2 += np.eye(sigma2.shape[0]) * epsilon
    
    diff = mu1 - mu2
    
    # Fixed: Handle linalg.sqrtm return value properly
    try:
        # Try computing matrix square root
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        
        # Handle complex values
        if np.iscomplexobj(covmean):
            if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
                print("Warning: Imaginary component in matrix square root is significant")
            covmean = covmean.real
            
    except Exception as e:
        print(f"Warning: Matrix square root computation failed: {e}")
        # Fallback: use identity matrix (will give higher FID)
        covmean = np.eye(sigma1.shape[0])
    
    # Compute FID
    fid = diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)
    
    # Ensure FID is non-negative
    return max(float(fid), 0.0)

# --- Alternative FID computation (more stable) ---
def compute_fid_stable(real_features, gen_features):
    """
    More numerically stable FID computation using eigenvalue decomposition.
    """
    mu1 = np.mean(real_features, axis=0)
    mu2 = np.mean(gen_features, axis=0)
    sigma1 = np.cov(real_features, rowvar=False)
    sigma2 = np.cov(gen_features, rowvar=False)
    
    # Add regularization
    epsilon = 1e-6
    sigma1 += np.eye(sigma1.shape[0]) * epsilon
    sigma2 += np.eye(sigma2.shape[0]) * epsilon
    
    diff = mu1 - mu2
    
    # Use eigenvalue decomposition for more stable computation
    try:
        # Compute sigma1^{1/2} * sigma2 * sigma1^{1/2}
        eigvals1, eigvecs1 = np.linalg.eigh(sigma1)
        eigvals1 = np.maximum(eigvals1, epsilon)  # Ensure positive eigenvalues
        sqrt_sigma1 = eigvecs1 @ np.diag(np.sqrt(eigvals1)) @ eigvecs1.T
        
        M = sqrt_sigma1 @ sigma2 @ sqrt_sigma1
        eigvals_M, _ = np.linalg.eigh(M)
        eigvals_M = np.maximum(eigvals_M, 0)  # Ensure non-negative eigenvalues
        
        trace_sqrt_M = np.sum(np.sqrt(eigvals_M))
        
        fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * trace_sqrt_M
        
    except Exception as e:
        print(f"Warning: Stable FID computation failed: {e}")
        # Fallback to simple distance
        fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2)
    
    return max(float(fid), 0.0)

# --- Load Trained Model from Best Checkpoint ---
def load_best_model(model_class, model_name, hidden, layers):
    best_path = os.path.join(cfg.load_dir, f"{model_name}_best.pt")
    if not os.path.exists(best_path):
        raise FileNotFoundError(f"Best checkpoint not found: {best_path}")
    
    model = model_class(in_channels=3, hidden=hidden, n_layers=layers)
    checkpoint = torch.load(best_path, map_location=cfg.device)
    model_state = checkpoint['model_state']
    
    # Handle DataParallel compatibility
    if isinstance(model, torch.nn.DataParallel):
        current_state = model.module.state_dict()
    else:
        # Adjust for DataParallel checkpoints
        if all(k.startswith('module.') for k in model_state.keys()):
            model_state = {k.replace('module.', ''): v for k, v in model_state.items()}
        current_state = model.state_dict()
    
    # Handle size mismatches, especially for RowLSTM state_to_state weights
    adjusted_state = {}
    for key, checkpoint_param in model_state.items():
        if key in current_state:
            current_param = current_state[key]
            
            # Handle specific size mismatch for RowLSTM state_to_state weights
            if ('state_to_state.weight' in key and 
                checkpoint_param.shape != current_param.shape and
                len(checkpoint_param.shape) == 4 and len(current_param.shape) == 4):
                
                # Expected: checkpoint [384, 96, 1, 1] -> model [384, 96, 3, 1]
                if (checkpoint_param.shape[2] == 1 and current_param.shape[2] == 3 and
                    checkpoint_param.shape[0] == current_param.shape[0] and
                    checkpoint_param.shape[1] == current_param.shape[1]):
                    
                    # Expand the checkpoint parameter by repeating along dimension 2
                    expanded_param = checkpoint_param.repeat(1, 1, 3, 1)
                    adjusted_state[key] = expanded_param
                   # print(f"Adjusted {key} from {checkpoint_param.shape} to {expanded_param.shape}")
                else:
                   # print(f"Warning: Cannot adjust {key} from {checkpoint_param.shape} to {current_param.shape}")
                    adjusted_state[key] = checkpoint_param
            
            # Handle other potential size mismatches
            elif checkpoint_param.shape != current_param.shape:
                print(f"Warning: Size mismatch for {key}: checkpoint {checkpoint_param.shape} vs model {current_param.shape}")
                # Try to handle common mismatches
                if checkpoint_param.numel() == current_param.numel():
                    # If total elements match, try reshaping
                    try:
                        adjusted_state[key] = checkpoint_param.reshape(current_param.shape)
                      #  print(f"Reshaped {key} from {checkpoint_param.shape} to {current_param.shape}")
                    except:
                        adjusted_state[key] = checkpoint_param
                else:
                    adjusted_state[key] = checkpoint_param
            else:
                adjusted_state[key] = checkpoint_param
        else:
            print(f"Warning: Key {key} from checkpoint not found in current model")
    
    # Load the adjusted state dict
    if isinstance(model, torch.nn.DataParallel):
        try:
            model.module.load_state_dict(adjusted_state, strict=False)
        except Exception as e:
            print(f"Warning: Error loading state dict for DataParallel model: {e}")
            model.module.load_state_dict(adjusted_state, strict=False)
    else:
        try:
            model.load_state_dict(adjusted_state, strict=False)
        except Exception as e:
            print(f"Warning: Error loading state dict: {e}")
            model.load_state_dict(adjusted_state, strict=False)
    
    model.to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    model.eval()
    print(f"Loaded best {model_name} model from {best_path}")
    return model

# --- Main Bonus Execution ---
def run_bonus_evaluation(num_gen_samples=8, num_real_samples=500, temperature=0.95, batch_size=8):
    # Clear GPU memory before starting
    torch.cuda.empty_cache()
    gc.collect()
    
    # Load Inception models
    inception_model = get_inception_model(cfg.device)
    feature_extractor = InceptionFeatureExtractor(cfg.device)
    
    # Load real data for FID (sample from validation set)
    val_ds = torchvision.datasets.CIFAR10(cfg.data_dir, train=False, download=True, transform=transforms.ToTensor())
    real_loader = DataLoader(val_ds, batch_size=cfg.val_batch, shuffle=True, num_workers=0)
    real_images = []
    for batch in real_loader:
        real_images.append(batch[0])
        if len(real_images) * cfg.val_batch >= num_real_samples:
            break
    real_images = torch.cat(real_images)[:num_real_samples]
    print(f"Loaded {len(real_images)} real images")
    
    real_features = extract_features(real_images, feature_extractor, cfg.device, batch_size=batch_size)
    print(f"Extracted features from {len(real_images)} real images")
    
    # Free memory after real feature extraction
    del real_images
    torch.cuda.empty_cache()
    gc.collect()
    
    # Models to evaluate - NOTE: You need to define these model classes
    models_config = [
        ("PixelCNN", PixelCNN, cfg.pixelcnn_hidden, cfg.pixelcnn_layers),
        ("RowLSTM", RowLSTM, cfg.hidden_size, cfg.lstm_layers),
        ("DiagonalBiLSTM_UltraFast", DiagonalBiLSTM, cfg.diagonal_hidden, cfg.diagonal_layers)
    ]
    
    results = {}
    for model_name, model_class, hidden, layers in models_config:
        print(f"\n{'='*50}\nEvaluating {model_name}\n{'='*50}")
        
        # Load model
        try:
            model = load_best_model(model_class, model_name, hidden, layers)
        except Exception as e:
            print(f"Error loading {model_name}: {e}")
            continue
        
        # Generate samples
        try:
            samples = generate_samples(model, num_samples=num_gen_samples, temperature=temperature, device=cfg.device)
            
            # Qualitative: Display samples
            display_samples(samples, model_name)
            
            # Quantitative: IS
            is_mean, is_std = compute_is(samples, inception_model, cfg.device, num_splits=4, batch_size=batch_size)
            print(f"Inception Score: {is_mean:.4f} ± {is_std:.4f}")
            
            # Quantitative: FID (use stable version)
            gen_features = extract_features(samples, feature_extractor, cfg.device, batch_size=batch_size)
            fid = compute_fid_stable(real_features, gen_features)  # Use stable version
            print(f"Fréchet Inception Distance (FID): {fid:.4f}")
            
            # Save generated samples
            torch.save(samples.cpu(), os.path.join(cfg.save_dir, f'{model_name}_generated_samples.pt'))
            
            # Store results
            results[model_name] = {
                'is_mean': is_mean,
                'is_std': is_std,
                'fid': fid
            }
            
        except Exception as e:
            print(f"Error evaluating {model_name}: {e}")
            results[model_name] = {
                'is_mean': 0.0,
                'is_std': 0.0,
                'fid': float('inf')
            }
        
        # Free memory after processing each model
        if 'model' in locals():
            del model
        if 'samples' in locals():
            del samples
        if 'gen_features' in locals():
            del gen_features
        torch.cuda.empty_cache()
        gc.collect()
    
    # Print comparison table
    print(f"\n{'='*50}\nModel Comparison\n{'='*50}")
    print(f"{'Model':<25} {'IS (mean ± std)':<20} {'FID':<10}")
    print("-" * 55)
    for model_name, metrics in results.items():
        if metrics['fid'] == float('inf'):
            fid_str = "Failed"
        else:
            fid_str = f"{metrics['fid']:.4f}"
        print(f"{model_name:<25} {metrics['is_mean']:.4f} ± {metrics['is_std']:.4f} {fid_str:<10}")
    
    # Also save results to file
    results_file = os.path.join(cfg.save_dir, 'evaluation_results.txt')
    with open(results_file, 'w') as f:
        f.write("Model Evaluation Results\n")
        f.write("=" * 50 + "\n")
        for model_name, metrics in results.items():
            f.write(f"{model_name}:\n")
            f.write(f"  Inception Score: {metrics['is_mean']:.4f} ± {metrics['is_std']:.4f}\n")
            f.write(f"  FID: {metrics['fid']:.4f}\n")
            f.write("\n")
    
    print(f"\nResults saved to: {results_file}")
    return results

# Run the bonus evaluation
if __name__ == "__main__":
    # Note: Make sure to include your model class definitions (PixelCNN, RowLSTM, DiagonalBiLSTM) 
    # before running this code
    results = run_bonus_evaluation(num_gen_samples=8, num_real_samples=500, temperature=0.95, batch_size=8)

In [ ]:
# ========================================
# Cell 4b: Bonus Part - Sample Generation, Qualitative Inspection, IS, and FID
# ========================================


import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- Fix save directory reference ---
try:
    save_dir = cfg.save_dir
except:
    save_dir = "/kaggle/working/pixelrnn_results"
    os.makedirs(save_dir, exist_ok=True)

# --- Sample Generation ---
@torch.no_grad()
def sample_from_pixelrnn_autoregressive(model, num_samples=32, temperature=0.8, top_k=32, device=None):
    """
    Generate samples using autoregressive sampling following paper methodology
    
    Args:
        model: Trained PixelRNN model
        num_samples: Number of samples to generate
        temperature: Sampling temperature (1.0 = standard, <1.0 = sharper, >1.0 = smoother)
        top_k: If set, use top-k sampling to reduce noise
        device: Device for computation
    
    Returns:
        Generated samples as uint8 tensor (num_samples, 3, 32, 32)
    """
    if device is None:
        device = next(model.parameters()).device
    
    model.eval()
    
    # Handle DataParallel wrapper
    if isinstance(model, torch.nn.DataParallel):
        model_fn = model.module
    else:
        model_fn = model
    
    # Initialize with zeros in [0,1] scale
    samples = torch.zeros(num_samples, 3, 32, 32, dtype=torch.float32, device=device)
    
    print(f"Generating {num_samples} samples with temperature {temperature} and top_k {top_k}...")
    
    # Autoregressive generation pixel by pixel, following RGB order within each pixel
    with tqdm(total=32*32*3, desc="Generating") as pbar:
        for i in range(32):  # Height
            for j in range(32):  # Width
                for c in range(3):  # Channels (R, G, B)
                    # Get logits from model (input already in [0,1])
                    logits = model_fn(samples)  # (batch, 3, 256, 32, 32)
                    
                    # Extract logits for current pixel and channel
                    pixel_channel_logits = logits[:, c, :, i, j]  # (batch, 256)
                    
                    # Apply temperature
                    pixel_channel_logits = pixel_channel_logits / temperature
                    
                    # Convert to probabilities
                    probs = F.softmax(pixel_channel_logits, dim=-1)
                    
                    if top_k is not None:
                        values, indices = torch.topk(probs, k=top_k, dim=-1)
                        probs = torch.zeros_like(probs).scatter_(-1, indices, values)
                        probs = probs / probs.sum(-1, keepdim=True)
                    
                    # Sample from categorical distribution (0-255)
                    sampled_values = torch.multinomial(probs, num_samples=1).squeeze(-1).float()
                    
                    # Update samples in [0,1] scale
                    samples[:, c, i, j] = sampled_values / 255.0
                    
                    pbar.update(1)
    
    # Convert to uint8
    return (samples.cpu() * 255).to(torch.uint8)

# --- Quality Metrics ---
def calculate_sample_diversity(samples, num_comparisons=500):
    """Calculate diversity of samples using pairwise L2 distances"""
    samples_flat = samples.float().view(samples.size(0), -1)
    
    # Randomly select pairs for comparison (reduced for efficiency)
    indices = torch.randperm(len(samples))[:min(num_comparisons, len(samples))]
    distances = []
    
    for i in range(0, len(indices)-1, 2):
        if i+1 < len(indices):
            dist = torch.norm(samples_flat[indices[i]] - samples_flat[indices[i+1]]).item()
            distances.append(dist)
    
    if not distances:
        return {
            'mean_distance': 0.0,
            'std_distance': 0.0,
            'min_distance': 0.0,
            'max_distance': 0.0
        }
    
    return {
        'mean_distance': np.mean(distances),
        'std_distance': np.std(distances),
        'min_distance': np.min(distances),
        'max_distance': np.max(distances)
    }

def analyze_color_distribution(samples):
    """Analyze color distribution properties of samples"""
    if samples.dtype == torch.uint8:
        samples_float = samples.float()
    else:
        samples_float = samples * 255.0
    
    # Per-channel statistics
    channel_stats = {}
    channel_names = ['Red', 'Green', 'Blue']
    
    for c, name in enumerate(channel_names):
        channel_data = samples_float[:, c, :, :].flatten()
        channel_stats[name] = {
            'mean': float(channel_data.mean()),
            'std': float(channel_data.std()),
            'min': float(channel_data.min()),
            'max': float(channel_data.max())
        }
    
    # Overall statistics
    all_pixels = samples_float.flatten()
    overall_stats = {
        'mean_intensity': float(all_pixels.mean()),
        'std_intensity': float(all_pixels.std()),
        'brightness_distribution': torch.histc(all_pixels, bins=50, min=0, max=255).numpy()
    }
    
    return channel_stats, overall_stats

# --- Image Completion ---
@torch.no_grad()
def demonstrate_image_completion_advanced(model, model_name, num_examples=6, mask_type='bottom_half', temperature=0.8, top_k=32):
    """
    Advanced image completion demonstration
    
    Args:
        model: Trained model
        model_name: Name for saving
        num_examples: Number of examples to complete
        mask_type: Type of occlusion ('bottom_half', 'random_patches', 'center_square')
        temperature Aluminum sampling temperature
        top_k: Top-k sampling parameter
    """
    print(f"\nDemonstrating image completion for {model_name} (mask: {mask_type})")
    
    # Get some validation images
    try:
        val_batch = next(iter(val_loader))[0][:num_examples]
        if hasattr(cfg, 'device'):
            val_batch = val_batch.to(cfg.device)
        else:
            val_batch = val_batch.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    except:
        print(f"Could not load validation data for completion demo")
        return None, None, None
    
    original_images = val_batch.clone()
    
    # Create masks based on type - assume val_batch in [0,1]
    if mask_type == 'bottom_half':
        masked_images = val_batch.clone()
        masked_images[:, :, 16:, :] = 0.0  # Fill with zeros
        completion_region = (slice(None), slice(None), slice(16, None), slice(None))
        
    elif mask_type == 'center_square':
        masked_images = val_batch.clone()
        masked_images[:, :, 8:24, 8:24] = 0.0  # Mask center 16x16 region
        completion_region = (slice(None), slice(None), slice(8, 24), slice(8, 24))
        
    else:  # random_patches
        masked_images = val_batch.clone()
        for b in range(num_examples):
            # Create 3-4 random square patches to mask
            num_patches = np.random.randint(3, 5)
            for _ in range(num_patches):
                h_start = np.random.randint(0, 24)
                w_start = np.random.randint(0, 24)
                masked_images[b, :, h_start:h_start+8, w_start:w_start+8] = 0.0
        completion_region = None
    
    model.eval()
    if isinstance(model, torch.nn.DataParallel):
        model_fn = model.module
    else:
        model_fn = model
    
    completions = masked_images.clone()
    
    if mask_type in ['bottom_half', 'center_square']:
        # Complete specific region
        h_slice = completion_region[2]
        w_slice = completion_region[3]
        h_range = range(h_slice.start, h_slice.stop)
        w_range = range(w_slice.start, w_slice.stop)
        
        with tqdm(total=len(h_range)*len(w_range)*3, desc=f"Completing {mask_type}") as pbar:
            for i in h_range:
                for j in w_range:
                    for c in range(3):
                        logits = model_fn(completions)
                        pixel_logits = logits[:, c, :, i, j]
                        pixel_logits = pixel_logits / temperature
                        probs = F.softmax(pixel_logits, dim=-1)
                        
                        if top_k is not None:
                            values, indices = torch.topk(probs, k=top_k, dim=-1)
                            probs = torch.zeros_like(probs).scatter_(-1, indices, values)
                            probs = probs / probs.sum(-1, keepdim=True)
                        
                        pixel_values = torch.multinomial(probs, num_samples=1).squeeze(-1).float()
                        completions[:, c, i, j] = pixel_values / 255.0
                        pbar.update(1)
    
    else:  # random_patches
        mask = (masked_images == 0.0).any(dim=1, keepdim=True).expand_as(masked_images)
        
        with tqdm(total=32*32*3, desc="Completing random patches") as pbar:
            for i in range(32):
                for j in range(32):
                    for c in range(3):
                        needs_completion = mask[:, c, i, j]
                        if needs_completion.any():
                            logits = model_fn(completions)
                            pixel_logits = logits[:, c, :, i, j]
                            pixel_logits = pixel_logits / temperature
                            probs = F.softmax(pixel_logits, dim=-1)
                            
                            if top_k is not None:
                                values, indices = torch.topk(probs, k=top_k, dim=-1)
                                probs = torch.zeros_like(probs).scatter_(-1, indices, values)
                                probs = probs / probs.sum(-1, keepdim=True)
                            
                            pixel_values = torch.multinomial(probs, num_samples=1).squeeze(-1).float()
                            completions[needs_completion, c, i, j] = pixel_values[needs_completion] / 255.0
                        pbar.update(1)
    
    # Create comparison visualization with upscaling
    upscale_factor = 4
    def upscale(tensor):
        return F.interpolate(tensor, scale_factor=upscale_factor, mode='bilinear', antialias=True)
    
    comparison_grid = torch.cat([
        upscale(original_images.cpu()),
        upscale(masked_images.cpu()), 
        upscale(completions.cpu())
    ], dim=0)
    
    grid = make_grid(comparison_grid, nrow=num_examples, padding=2 * upscale_factor, pad_value=1)
    
    plt.figure(figsize=(20, 12))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.title(f'{model_name} - Image Completion ({mask_type})\n'
              f'Top: Original | Middle: Masked | Bottom: Completed', fontsize=14, fontweight='bold')
    plt.axis('off')
    
    try:
        save_path = os.path.join(save_dir, f'{model_name}_completion_{mask_type}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Completion results saved to: {save_path}")
    except Exception as e:
        print(f"Could not save completion results: {e}")
    
    plt.show()
    
    return original_images, masked_images, completions

# --- Modified Comprehensive Model Evaluation ---
def comprehensive_model_evaluation(model_configs, num_samples=32):
    """
    Run comprehensive evaluation on all trained models
    
    Args:
        model_configs: List of (model_class, model_name) tuples
        num_samples: Number of samples to generate for evaluation
    """
    print("="*100)
    print("COMPREHENSIVE BONUS EVALUATION - ADVANCED ANALYSIS")
    print("="*100)
    
    all_results = {}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    for model_class, model_name in model_configs:
        print(f"\n{'='*60}")
        print(f"EVALUATING {model_name.upper()}")
        print(f"{'='*60}")
        
        # Check for different possible checkpoint names
        possible_paths = [
            os.path.join(save_dir, f"{model_name}_best.pt"),
            os.path.join(save_dir, f"{model_name}_UltraFast_best.pt"),
            os.path.join(save_dir, f"DiagonalBiLSTM_UltraFast_best.pt") if "Diagonal" in model_name else None
        ]
        possible_paths = [p for p in possible_paths if p is not None]
        
        best_path = None
        for path in possible_paths:
            if os.path.exists(path):
                best_path = path
                break
        
        if best_path is None:
            print(f"No checkpoint found for {model_name}, skipping...")
            all_results[model_name] = {'error': 'No checkpoint found'}
            continue
        
        # Load model with correct parameters
        print(f"Loading model from: {best_path}")
        try:
            if 'PixelCNN' in model_name:
                model = model_class(in_channels=3, hidden=cfg.pixelcnn_hidden, n_layers=cfg.pixelcnn_layers)
            elif 'RowLSTM' in model_name:
                model = model_class(in_channels=3, hidden=cfg.hidden_size, n_layers=cfg.lstm_layers)
            elif 'Diagonal' in model_name:
                model = model_class(in_channels=3, hidden=cfg.diagonal_hidden, n_layers=cfg.diagonal_layers)
            else:
                model = model_class()
                
            model = model.to(device)
        except Exception as e:
            print(f"Failed to create model {model_name}: {e}")
            all_results[model_name] = {'error': f'Model creation failed: {e}'}
            continue
        
        # Enable multi-GPU if available
        if torch.cuda.device_count() > 1:
            model = torch.nn.DataParallel(model)
        
        # Load checkpoint with compatibility fix for RowLSTM
        try:
            checkpoint = torch.load(best_path, map_location=device)
            model_state_dict = checkpoint['model_state']
            
            if 'RowLSTM' in model_name:
                # Adjust state_dict for RowLSTM to match expected [384, 96, 3, 1]
                new_state_dict = {}
                for key, value in model_state_dict.items():
                    if 'lstm_layers' in key and 'state_to_state.weight' in key:
                        # Checkpoint has shape [384, 96, 1, 1], model expects [384, 96, 3, 1]
                        # Extend the weight to [384, 96, 3, 1] by repeating the 1x1 kernel across 3 channels
                        if value.shape == torch.Size([384, 96, 1, 1]):
                            value = value.repeat(1, 1, 3, 1)  # Repeat along the height dimension
                            print(f"Adjusted {key} from {value.shape} to [384, 96, 3, 1]")
                        new_state_dict[key] = value
                    else:
                        new_state_dict[key] = value
                
                model_state_dict = new_state_dict
            
            if isinstance(model, torch.nn.DataParallel):
                model.module.load_state_dict(model_state_dict)
            else:
                model.load_state_dict(model_state_dict)
            
            model.eval()
        except Exception as e:
            print(f"Failed to load checkpoint for {model_name}: {e}")
            all_results[model_name] = {'error': f'Checkpoint loading failed: {e}'}
            continue
        
        results = {
            'model_name': model_name,
            'best_val_bits': checkpoint.get('best_val_bits', 'N/A'),
            'best_val_nll': checkpoint.get('best_val_nll', 'N/A')
        }
        
        # 1. Sample Generation
        print("\n1. SAMPLE GENERATION")
        print("-" * 30)
        
        temperatures = [0.8]  # Focus on lower temperature for sharper samples
        sample_results = {}
        
        for temp in temperatures:
            print(f"Generating samples at temperature {temp}...")
            try:
                samples = sample_from_pixelrnn_autoregressive(
                    model, num_samples=num_samples, temperature=temp, top_k=32, device=device
                )
                sample_results[f'temp_{temp}'] = samples
                
                # For better visualization, upscale samples
                upscale_factor = 4
                upscaled_samples = F.interpolate(samples.float() / 255.0, scale_factor=upscale_factor, mode='bilinear', antialias=True)
                
                # Save sample grid
                grid = make_grid(upscaled_samples, nrow=8, padding=2 * upscale_factor, pad_value=1)
                plt.figure(figsize=(20, 20))
                plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
                plt.title(f'{model_name} - Generated Samples (Temperature = {temp})', 
                         fontsize=14, fontweight='bold')
                plt.axis('off')
                
                try:
                    save_path = os.path.join(save_dir, f'{model_name}_samples_temp_{temp}.png')
                    plt.savefig(save_path, dpi=150, bbox_inches='tight')
                    print(f"Samples saved to: {save_path}")
                except:
                    print("Could not save samples")
                
                plt.show()
                
            except Exception as e:
                print(f"Sample generation failed for temperature {temp}: {e}")
                sample_results[f'temp_{temp}'] = None
        
        # 2. Sample Quality Analysis
        print("\n2. SAMPLE QUALITY ANALYSIS")
        print("-" * 30)
        
        if sample_results.get('temp_0.8') is not None:
            samples = sample_results['temp_0.8']
            
            # Diversity analysis
            try:
                diversity_metrics = calculate_sample_diversity(samples)
                print(f"Sample diversity metrics:")
                for key, value in diversity_metrics.items():
                    print(f"  {key}: {value:.4f}")
                    results[f'diversity_{key}'] = value
            except Exception as e:
                print(f"Diversity analysis failed: {e}")
            
            # Color distribution analysis
            try:
                channel_stats, overall_stats = analyze_color_distribution(samples)
                print(f"\nColor distribution analysis:")
                for channel, stats in channel_stats.items():
                    print(f"  {channel}: mean={stats['mean']:.1f}, std={stats['std']:.1f}")
                    results[f'{channel.lower()}_mean'] = stats['mean']
                    results[f'{channel.lower()}_std'] = stats['std']
                
                print(f"  Overall intensity: {overall_stats['mean_intensity']:.1f} ± {overall_stats['std_intensity']:.1f}")
                results['overall_mean_intensity'] = overall_stats['mean_intensity']
                results['overall_std_intensity'] = overall_stats['std_intensity']
                
                # Save brightness histogram
                plt.figure(figsize=(10, 6))
                plt.bar(range(len(overall_stats['brightness_distribution'])), 
                       overall_stats['brightness_distribution'], alpha=0.7)
                plt.title(f'{model_name} - Brightness Distribution of Generated Samples')
                plt.xlabel('Pixel Intensity (0-255)')
                plt.ylabel('Frequency')
                plt.grid(True, alpha=0.3)
                
                try:
                    hist_save_path = os.path.join(save_dir, f'{model_name}_brightness_histogram.png')
                    plt.savefig(hist_save_path, dpi=150, bbox_inches='tight')
                except:
                    print("Could not save histogram")
                    
                plt.show()
            except Exception as e:
                print(f"Color distribution analysis failed: {e}")
        
        # 3. Image Completion Demonstrations
        print("\n3. IMAGE COMPLETION DEMONSTRATIONS")
        print("-" * 40)
        
        completion_types = ['bottom_half', 'center_square']  # Reduced for efficiency
        completion_results = {}
        
        for comp_type in completion_types:
            try:
                print(f"Demonstrating {comp_type} completion...")
                original, masked, completed = demonstrate_image_completion_advanced(
                    model, model_name, num_examples=6, mask_type=comp_type, temperature=0.8, top_k=32
                )
                
                if original is not None:
                    completion_results[comp_type] = {
                        'original': original,
                        'masked': masked, 
                        'completed': completed
                    }
                    
                    # Calculate completion quality metrics
                    if comp_type == 'bottom_half':
                        original_region = original[:, :, 16:, :].float()
                        completed_region = completed[:, :, 16:, :].float()
                        mse = torch.mean((original_region - completed_region) ** 2).item()
                        results[f'{comp_type}_mse'] = mse
                        print(f"  MSE for {comp_type}: {mse:.4f}")
                
            except Exception as e:
                print(f"Completion demonstration failed for {comp_type}: {e}")
                completion_results[comp_type] = None
        
        # 4. Performance Summary
        print("\n4. PERFORMANCE SUMMARY")
        print("-" * 25)
        
        if isinstance(results['best_val_bits'], (int, float)):
            print(f"Best validation bits/dim: {results['best_val_bits']:.6f}")
            if 'best_val_nll' in results and isinstance(results['best_val_nll'], (int, float)):
                print(f"Best validation NLL: {results['best_val_nll']:.6f}")
        
        print(f"Sample generation: {'✓ Success' if sample_results.get('temp_0.8') is not None else '✗ Failed'}")
        print(f"Image completion: {'✓ Success' if any(completion_results.values()) else '✗ Failed'}")
        
        # Assign overall grade
        grade_factors = []
        if isinstance(results.get('best_val_bits'), (int, float)):
            if results['best_val_bits'] < 3.5:
                grade_factors.append('A')
            elif results['best_val_bits'] < 4.0:
                grade_factors.append('B') 
            else:
                grade_factors.append('C')
        
        if sample_results.get('temp_0.8') is not None:
            grade_factors.append('A')
        else:
            grade_factors.append('C')
            
        if any(completion_results.values()):
            grade_factors.append('A')
        else:
            grade_factors.append('C')
        
        # Convert grades to numeric and average
        if grade_factors:
            grade_map = {'A': 4, 'B': 3, 'C': 2, 'D': 1, 'F': 0}
            avg_grade = np.mean([grade_map[g] for g in grade_factors])
            
            if avg_grade >= 3.5:
                overall_grade = 'A'
            elif avg_grade >= 2.5:
                overall_grade = 'B'
            elif avg_grade >= 1.5:
                overall_grade = 'C'
            else:
                overall_grade = 'D'
        else:
            overall_grade = 'F'
        
        results['overall_grade'] = overall_grade
        print(f"Overall evaluation grade: {overall_grade}")
        
        all_results[model_name] = results
        
        # Clean up memory
        del model
        torch.cuda.empty_cache()
    
    # 6. Final Comparative Analysis
    print("\n" + "="*100)
    print("FINAL COMPARATIVE BONUS EVALUATION RESULTS")
    print("="*100)
    
    # Create comprehensive results table
    comparison_data = {
        'Model': [],
        'Val Bits/dim': [],
        'Sample Quality': [],
        'Completion Success': [],
        'Overall Grade': []
    }
    
    for model_name, results in all_results.items():
        if 'error' in results:
            continue
            
        comparison_data['Model'].append(model_name)
        
        if isinstance(results.get('best_val_bits'), (int, float)):
            comparison_data['Val Bits/dim'].append(f"{results['best_val_bits']:.6f}")
        else:
            comparison_data['Val Bits/dim'].append('N/A')
            
        if 'diversity_mean_distance' in results:
            comparison_data['Sample Quality'].append(f"{results['diversity_mean_distance']:.2f}")
        else:
            comparison_data['Sample Quality'].append('N/A')
            
        comparison_data['Completion Success'].append('✓' if results.get('bottom_half_mse') is not None else '✗')
        comparison_data['Overall Grade'].append(results.get('overall_grade', 'N/A'))
    
    if comparison_data['Model']:
        df_bonus = pd.DataFrame(comparison_data)
        print("\nBonus Evaluation Summary Table:")
        print("-" * 80)
        print(df_bonus.to_string(index=False))
        
        # Save comprehensive results
        try:
            df_bonus.to_csv(os.path.join(save_dir, 'bonus_evaluation_results.csv'), index=False)
            print(f"\nResults saved to: {save_dir}")
        except:
            print("Could not save results CSV")
    else:
        print("No models were successfully evaluated.")
        df_bonus = pd.DataFrame()
    
    print(f"\nAll bonus evaluation results attempted.")
    print("Advanced evaluation complete!")
    
    return all_results, df_bonus

# --- Execute Comprehensive Bonus Evaluation ---
def run_full_bonus_evaluation():
    """Execute the complete bonus evaluation pipeline"""
    
    model_configurations = [
        (PixelCNN, "PixelCNN"),
        (RowLSTM, "RowLSTM"), 
        (DiagonalBiLSTM, "DiagonalBiLSTM")
    ]
    
    print("Starting comprehensive bonus evaluation...")
    print("This includes:")
    print("- Autoregressive sample generation")
    print("- Sample quality and diversity analysis")  
    print("- Image completion demonstrations")
    print("- Color distribution analysis")
    print("- Comprehensive comparative visualization")
    
    try:
        results, summary_df = comprehensive_model_evaluation(
            model_configurations, 
            num_samples=32  # Optimized for efficiency
        )
        
        return results, summary_df
    except Exception as e:
        print(f"Bonus evaluation failed: {e}")
        return {}, pd.DataFrame()

# Execute bonus evaluation
print("Bonus evaluation setup complete!")
print("Running full bonus evaluation...")
bonus_results, bonus_summary = run_full_bonus_evaluation()
print("Bonus evaluation complete!")